In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Diabetic Retinopathy Detection and Grading System\n",
    "\n",
    "This notebook demonstrates the end-to-end workflow for retinal image preprocessing, hybrid deep feature extraction with InceptionV3 + ResNet50, classical classifier training, and Grad-CAM visualization."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Configuration\n",
    "\n",
    "Update the dataset paths below to match your local APTOS 2019 or EyePACS dataset."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "CSV_PATH = 'dataset/train.csv'\n",
    "IMAGE_DIR = 'dataset/train_images'\n",
    "IMAGE_COL = 'id_code'\n",
    "LABEL_COL = 'diagnosis'\n",
    "IMAGE_EXT = '.png'\n",
    "EPOCHS = 15\n",
    "BATCH_SIZE = 16"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pathlib import Path\n",
    "\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from classifiers import train_and_evaluate_classifiers\n",
    "from feature_extraction import (\n",
    "    HybridTrainingConfig,\n",
    "    build_hybrid_model,\n",
    "    extract_fused_features,\n",
    "    plot_training_curves,\n",
    "    train_hybrid_model,\n",
    ")\n",
    "from gradcam import make_gradcam_heatmap, overlay_heatmap\n",
    "from preprocessing import (\n",
    "    CLASS_NAMES,\n",
    "    dataframe_to_dataset,\n",
    "    load_preprocessed_arrays,\n",
    "    read_dataset,\n",
    "    split_dataframe,\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Load and Split Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df = read_dataset(CSV_PATH)\n",
    "train_df, test_df = split_dataframe(df, label_col=LABEL_COL)\n",
    "train_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Create TensorFlow Datasets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "train_ds = dataframe_to_dataset(\n",
    "    train_df,\n",
    "    image_dir=IMAGE_DIR,\n",
    "    image_col=IMAGE_COL,\n",
    "    label_col=LABEL_COL,\n",
    "    image_ext=IMAGE_EXT,\n",
    "    batch_size=BATCH_SIZE,\n",
    "    training=True,\n",
    ")\n",
    "\n",
    "test_ds = dataframe_to_dataset(\n",
    "    test_df,\n",
    "    image_dir=IMAGE_DIR,\n",
    "    image_col=IMAGE_COL,\n",
    "    label_col=LABEL_COL,\n",
    "    image_ext=IMAGE_EXT,\n",
    "    batch_size=BATCH_SIZE,\n",
    "    training=False,\n",
    "    shuffle=False,\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Train Hybrid Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "config = HybridTrainingConfig(epochs=EPOCHS)\n",
    "hybrid_model, feature_model, last_conv_layer_name = build_hybrid_model(config)\n",
    "history = train_hybrid_model(\n",
    "    hybrid_model,\n",
    "    train_ds=train_ds,\n",
    "    val_ds=test_ds,\n",
    "    y_train=train_df[LABEL_COL].to_numpy(),\n",
    "    epochs=EPOCHS,\n",
    ")\n",
    "plot_training_curves(history)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Extract Hybrid Features and Train Classical Classifiers"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train = extract_fused_features(feature_model, train_ds)\n",
    "X_test = extract_fused_features(feature_model, test_ds)\n",
    "y_train = train_df[LABEL_COL].to_numpy()\n",
    "y_test = test_df[LABEL_COL].to_numpy()\n",
    "\n",
    "results_df = train_and_evaluate_classifiers(\n",
    "    X_train,\n",
    "    y_train,\n",
    "    X_test,\n",
    "    y_test,\n",
    "    class_names=CLASS_NAMES,\n",
    ")\n",
    "results_df"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Grad-CAM Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "sample_images, sample_labels, _ = load_preprocessed_arrays(\n",
    "    test_df.head(1),\n",
    "    image_dir=IMAGE_DIR,\n",
    "    image_col=IMAGE_COL,\n",
    "    label_col=LABEL_COL,\n",
    "    image_ext=IMAGE_EXT,\n",
    ")\n",
    "\n",
    "heatmap = make_gradcam_heatmap(sample_images[0], hybrid_model, last_conv_layer_name)\n",
    "overlay = overlay_heatmap(heatmap, sample_images[0])\n",
    "\n",
    "plt.figure(figsize=(12, 4))\n",
    "plt.subplot(1, 3, 1)\n",
    "plt.imshow(sample_images[0])\n",
    "plt.title('Preprocessed Image')\n",
    "plt.axis('off')\n",
    "\n",
    "plt.subplot(1, 3, 2)\n",
    "plt.imshow(heatmap, cmap='jet')\n",
    "plt.title('Grad-CAM Heatmap')\n",
    "plt.axis('off')\n",
    "\n",
    "plt.subplot(1, 3, 3)\n",
    "plt.imshow(overlay)\n",
    "plt.title(f\"Overlay: {CLASS_NAMES[int(sample_labels[0])]}\")\n",
    "plt.axis('off')\n",
    "plt.tight_layout()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}